In [1]:
import pandas as pd

df = pd.read_csv("../data/tickets_final_cleaned.csv",
                 parse_dates=["Created_Date", "Resolved_Date"])

print(df.shape)
df.dtypes

(10000, 18)


Ticket_ID                        object
Customer_ID                      object
Product_ID                       object
Agent_ID                         object
Created_Date             datetime64[ns]
Resolved_Date            datetime64[ns]
Priority                         object
Channel                          object
Issue_Type                       object
Status                           object
SLA_Target_Hours                  int64
Resolution_Time_Hours           float64
SLA_Status                       object
Customer_Satisfaction           float64
CSAT_Missing_Flag                object
SLA_Mismatch_Flag                object
Elapsed_Hours                   float64
Channel_Missing_Flag             object
dtype: object

In [2]:
# only resolved tickets
resolved = df[df["Status"] == "Resolved"]

# Overall SLA status
print(resolved["SLA_Status"].value_counts())
print(resolved["SLA_Status"].value_counts(normalize=True) * 100)

SLA_Status
Breached    5193
Met         3234
Name: count, dtype: int64
SLA_Status
Breached    61.623354
Met         38.376646
Name: proportion, dtype: float64


In [3]:
breach_by_priority = (
    resolved.groupby("Priority")["SLA_Status"]
    .apply(lambda s: (s == "Breached").mean() * 100)
    .round(1)
    .sort_values(ascending=False)
)
print(breach_by_priority)

Priority
Critical    96.2
High        93.1
Medium      53.3
Low         32.8
Name: SLA_Status, dtype: float64


In [4]:
res_time = (
    resolved.groupby("Priority")["Resolution_Time_Hours"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)
res_time["Standard_Target"] = res_time.index.map({"Critical": 4, "High": 8, "Medium": 24, "Low": 48})
print(res_time)

          count   mean  median  min     max  Standard_Target
Priority                                                    
Critical    480  11.93   10.75  0.5   37.13                4
High       2042  16.78   15.93  0.5   51.00                8
Low        1547  42.23   41.53  0.5  106.57               48
Medium     4358  25.36   24.74  0.5   69.16               24


In [5]:
resolved.groupby("Priority")["Resolution_Time_Hours"].median()

Priority
Critical    10.750
High        15.935
Low         41.530
Medium      24.740
Name: Resolution_Time_Hours, dtype: float64